# 04 — Train adv-NO with Gaussian-center spatial loss (magic window)

Same data, perceptual/adversarial schedule, and train/val settings as [04_train_adv_no_magic_window.ipynb](04_train_adv_no_magic_window.ipynb), but with **`gaussian_center`** spatial loss (σ=32) from [07_train_l1_gauss_circles.ipynb](../07_train_l1_gauss_circles.ipynb).

Trains adv-NO on **2013–2022** data exported by [01_export_magic_window.ipynb](01_export_magic_window.ipynb). Rollout eval: [05_evaluate_adv_no_magic_window_rollout_gaussian.ipynb](05_evaluate_adv_no_magic_window_rollout_gaussian.ipynb).

**Change only `EXPORT_KEY`** in the setup cell (same window for train and eval).

| Item | Value |
|---|---|
| Data root | `Magic_window/data/{EXPORT_KEY}/` |
| Train export | `sample_13_22_train_uv_dt360` |
| Val export | merged at train time (random 70/30 split) |
| Test export (nb 05) | `sample_23_uv_dt360` |
| Run dir | `Models/adv_no/adv_no_uv_13_22_{EXPORT_KEY}_dt360_gauss_vgg_warm10_adv60` |

**Spatial loss:** `gaussian_center` (σ=32 px) — center errors weighted more than outer rings (`state⊙G`).

**Adv-NO schedule (same as notebook 04):** L1 ep 1–10, +VGG-19 ep 11–30, +adversarial ep 31+. Val metric: masked **RMSE**. AdamW `lr=2e-4`, `weight_decay=1e-4` (decoupled L2; toggle `OPTIMIZER` / `WEIGHT_DECAY` in setup).

**Viz:** Gaussian-circle validation plots in `viz/samples_gauss_circles/` + `viz/curves/circle_l1_overview.png`.

**Run order:** setup → export check → dataset stats → train config → preview → train → loss curves.

In [ ]:
import json
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Magic_window" else NOTEBOOK_DIR / "Magic_window"
ROOT = MW_DIR.parent
if str(MW_DIR) not in sys.path:
    sys.path.insert(0, str(MW_DIR))

from window_paths import train_prefix, val_prefix, test_prefix, split_manifest_path, window_data_dir
from run_registry import (
    GAUSS_RUN_TAG,
    collect_dataset_stats,
    run_dir,
    run_name,
    weight_decay_ablation_tags,
    weight_decay_run_tag,
    save_dataset_stats,
    update_comparison_index,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = MW_DIR / "data"
MODELS_DIR = MW_DIR / "Models" / "adv_no"
LOG_DIR = MW_DIR / "logs"
PNG_DIR = MW_DIR / "PNG"
for _d in (DATA_DIR, MODELS_DIR, LOG_DIR, PNG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

import importlib
import src.train.fix_window_train as _fwt
import src.train.fix_window_viz as _fwv
import src.train.losses as _losses
import src.train.adv_no_train as _adv
importlib.reload(_losses)
importlib.reload(_fwt)
importlib.reload(_fwv)
importlib.reload(_adv)

from src.train.adv_no_train import AdvNOTrainConfig, run_adv_no_training
from src.train.fix_window_train import (
    load_merged_training_arrays,
    load_training_arrays,
    resolve_train_val_indices,
    prepare_train_val_tensors,
)


def load_notebook_training_arrays(cfg):
    """Load arrays for preview/config cells; mirrors run_adv_no_training."""
    if cfg.merge_val_export_prefix and not cfg.val_data_prefix:
        return load_merged_training_arrays(cfg), None
    data, mask, day, hour, crop_h, crop_w = load_training_arrays(cfg)
    if cfg.val_data_prefix:
        val_tensors = load_training_arrays(cfg, data_prefix=cfg.val_data_prefix)
        return (data, mask, day, hour, crop_h, crop_w), val_tensors
    return (data, mask, day, hour, crop_h, crop_w), None

# --- ONLY CHANGE THIS when switching datasets ---
EXPORT_KEY = "legacy_smallfixwin_45deg"
# EXPORT_KEY = "magicwin_reference"  # e.g. magicwin_high_land, legacy_smallfixwin_45deg

DT = 360
DATA_PREFIX = train_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
VAL_DATA_PREFIX = val_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
TEST_DATA_PREFIX = test_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
EPOCHS = 100
MAX_SAMPLES = None  # use all exported sequences

# --- optimizer / L2 regularization ---
OPTIMIZER = "adamw"  # "adam" | "adamw" — AdamW = decoupled L2 (recommended with weight_decay)
WEIGHT_DECAY = 1e-4  # ablation grid: 0 | 1e-4 | 3e-4

# --- weight-decay ablation (see markdown cell below) ---
ABLATION_SUFFIX_RUN_TAG = False  # set True for wd ablation (arms get _wd* suffix)
BASE_RUN_TAG = GAUSS_RUN_TAG
if ABLATION_SUFFIX_RUN_TAG:
    RUN_TAG = weight_decay_run_tag(BASE_RUN_TAG, WEIGHT_DECAY)
else:
    RUN_TAG = BASE_RUN_TAG
RUN_NAME = run_name(EXPORT_KEY, dt=DT, run_tag=RUN_TAG)
RUN_DIR = run_dir(MODELS_DIR, EXPORT_KEY, dt=DT, run_tag=RUN_TAG)

# --- Gaussian-center spatial loss (see 07_train_l1_gauss_circles.ipynb) ---
LOSS_SPATIAL_MODE = "gaussian_center"
LOSS_GAUSSIAN_SIGMA = 32.0

LOSS_MODE = "adv_no"
LOSS_METRIC = "components"
CHANNEL_NAMES = ("u", "v")
iD = 2
BETA = 0.1
LAMBDA_PERC = 0.1
PERCEPTUAL_BACKEND = "vgg"
PERCEPTUAL_WEIGHTS = (0.1, 0.1, 1.0, 1.0, 1.0)
LPIPS_NET = "alex"
PERCEPTUAL_WARMUP_EPOCHS = 10
WARMUP_EPOCHS = 30
D_LR_RATIO = 0.5
D_SKIP_THRESHOLD = 0.2
USE_COSINE_LR = True
VAL_SPLIT = 0.3  # 30% validation, 70% train
VAL_SPLIT_MODE = "chronological"  # first 70% train, last 30% val by (day, hour)
MERGE_VAL_EXPORT = True  # merge train+val .npy exports before split
VAL_RECONSTRUCTION_METRIC = "rmse"  # best checkpoint + val curves use masked uv RMSE
# VAL_SPLIT_MODE = "random"  # random 70/30 split (seed below)
# VAL_RECONSTRUCTION_METRIC = "l1"
# VAL_SPLIT_MODE = "export"  # use random export-time split from 01_export_magic_window.ipynb
# MERGE_VAL_EXPORT = False
VAL_SPLIT_SEED = 42
DETACHED = False
SHOW_INLINE_PLOTS = False
PREVIEW_LOSS_MASK = True
PREVIEW_VAL_MASK = True
PREVIEW_VAL_MASK_SAMPLES = 3
PROTECT_EXISTING_RUN = False
FILTER_FULLY_OCEAN_SAMPLES = False
DEVICE = "cuda:7"
VIZ_EVERY = 1
VIZ_SAMPLES = 1
VIZ_SEED = 42
VIZ_RESIDUAL_METRIC = "l1"
VIZ_GAUSSIAN_CIRCLES = True
VIZ_GAUSSIAN_CIRCLE_RADII = (16.0, 32.0, 64.0)
VIZ_GAUSSIAN_INNER_CIRCLE_RADII = (32.0, 16.0)
VIZ_GAUSSIAN_FIG_SCALE = 2.6

print("EXPORT_KEY:", EXPORT_KEY)
print("DATA_PREFIX:", DATA_PREFIX)
print("VAL_DATA_PREFIX:", VAL_DATA_PREFIX)
print("TEST_DATA_PREFIX:", TEST_DATA_PREFIX)
print("RUN_NAME:", RUN_NAME)
print("RUN_DIR:", RUN_DIR)
print("epochs:", EPOCHS)
print(f"Optimizer: {OPTIMIZER}, lr=2e-4, weight_decay={WEIGHT_DECAY}")
print("val split:", f"{int((1 - VAL_SPLIT) * 100)}% train / {int(VAL_SPLIT * 100)}% val", f"({VAL_SPLIT_MODE}, seed={VAL_SPLIT_SEED})")
print("val metric:", VAL_RECONSTRUCTION_METRIC)
print("max_samples:", MAX_SAMPLES, "(all export sequences)")
print("loss spatial:", LOSS_SPATIAL_MODE, f"(sigma={LOSS_GAUSSIAN_SIGMA})")
print("rollout eval:", MW_DIR / "05_evaluate_adv_no_magic_window_rollout_gaussian.ipynb")

## Weight-decay ablation (optional)

Compare **AdamW + L2** on the **same train/val split** (`VAL_SPLIT_SEED = 42`, random 70/30).

| Arm | `WEIGHT_DECAY` | Run tag suffix | Example run dir |
|-----|----------------|----------------|-----------------|
| A | `0` | `_wd0` | `.../gauss_gauss_vgg_warm10_adv60_wd0/` |
| B | `1e-4` | `_wd1e-4` | `.../gauss_vgg_warm10_adv60_wd1e-4/` |
| C | `3e-4` | `_wd3e-4` | `.../gauss_vgg_warm10_adv60_wd3e-4/` |

**How to run**

1. Set `ABLATION_SUFFIX_RUN_TAG = True` so each `WEIGHT_DECAY` gets its own folder.
2. Run the notebook **three times**, changing only `WEIGHT_DECAY` each time (0 → 1e-4 → 3e-4).
3. Leave `VAL_SPLIT_SEED`, `EXPORT_KEY`, schedule, and `OPTIMIZER` unchanged.
4. After all runs finish, run the **Compare weight-decay ablation** cell below for overlaid val RMSE curves.

**What to look for**

- Lower **best val RMSE** with similar or lower train L1 → less overfitting.
- If val RMSE rises at `3e-4`, the model may be under-regularized at `1e-4` or over-regularized at `3e-4`.
- Rollout (+15/+30/+45d) can be evaluated in notebook 05 for each run tag (point `RUN_TAG` / model dir at the matching `_wd*` folder).


In [ ]:
assert "DATA_PREFIX" in globals(), "Run the setup cell above first."
print("Ready:", DATA_PREFIX)
print("Val export:", VAL_DATA_PREFIX)
print("Next: export check → train config → preview → train")


In [ ]:
import numpy as np

if "DATA_PREFIX" not in globals():
    raise RuntimeError("Run the setup cell above first.")

for label, prefix in (("train", DATA_PREFIX), ("val", VAL_DATA_PREFIX)):
    sims_path = Path(f"{prefix}_sims.npy")
    if not sims_path.exists():
        raise FileNotFoundError(
            f"Missing {label} export: {sims_path}\n"
            "Run Magic_window/01_export_magic_window.ipynb (train + val exports)."
        )

train_sims = np.load(f"{DATA_PREFIX}_sims.npy", mmap_mode="r")
val_sims = np.load(f"{VAL_DATA_PREFIX}_sims.npy", mmap_mode="r")
mask = np.load(f"{DATA_PREFIX}_mask.npy", mmap_mode="r")
window = np.load(f"{DATA_PREFIX}_window.npy")
land_frac = float(mask.mean())
static_mask = len(mask) <= 1 or np.all(mask == mask[0:1])
n_train_use = len(train_sims) if MAX_SAMPLES is None else min(int(MAX_SAMPLES), len(train_sims))

print(f"Train export: {DATA_PREFIX}")
print(f"  sims shape: {train_sims.shape}  (N, T, C, H, W)")
print(f"  sequences for training: {n_train_use} / {len(train_sims)}")
print(f"Val export: {VAL_DATA_PREFIX}")
print(f"  sims shape: {val_sims.shape}")
print(f"  mask: {land_frac:.2%} land | static={static_mask}")
print(f"  window [top, left, h, w, angle]: {window}")
if MERGE_VAL_EXPORT and VAL_SPLIT_MODE != "export":
    n_merged = len(train_sims) + len(val_sims)
    n_train_split = int(n_merged * (1.0 - VAL_SPLIT))
    n_val_split = n_merged - n_train_split
    print(
        f"  training split: merge {len(train_sims)}+{len(val_sims)}={n_merged} → "
        f"{n_train_split} train + {n_val_split} val "
        f"({VAL_SPLIT_MODE}, {VAL_SPLIT:.0%} val, seed={VAL_SPLIT_SEED})"
    )
else:
    print(
        f"  export-time split: {n_train_use} train + {len(val_sims)} val "
        f"(see {EXPORT_KEY}/sample_13_22_split.json)"
    )


In [ ]:
# Collect and save dataset statistics for cross-dataset comparison
assert "EXPORT_KEY" in globals(), "Run the setup cell first."

RUN_DIR.mkdir(parents=True, exist_ok=True)
dataset_stats = collect_dataset_stats(DATA_DIR, EXPORT_KEY, dt=DT)
stats_path = save_dataset_stats(RUN_DIR, dataset_stats)
index_path = update_comparison_index(
    MODELS_DIR,
    run_name_value=RUN_NAME,
    export_key=EXPORT_KEY,
    run_dir_path=RUN_DIR,
    dataset_stats=dataset_stats,
)

print(f"Saved dataset stats: {stats_path}")
print(
    f"  train {dataset_stats['train']['n_sequences']} | "
    f"val {dataset_stats['val']['n_sequences']} | "
    f"test {dataset_stats['test']['n_sequences']} | "
    f"land {dataset_stats['train']['land_fraction']:.2%}"
)
print(f"Updated comparison index: {index_path}")


In [ ]:
train_config = AdvNOTrainConfig(
    data_prefix=DATA_PREFIX,
    run_name=RUN_NAME,
    dt_hours=DT,
    epochs=EPOCHS,
    batch_size=4,
    lr=2e-4,
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    d_lr=1e-4,
    d_lr_follow_g=True,
    d_lr_ratio=D_LR_RATIO,
    d_skip_threshold=D_SKIP_THRESHOLD,
    use_cosine_lr=USE_COSINE_LR,
    iD=iD,
    device=DEVICE,
    loss_mode=LOSS_MODE,
    loss_metric=LOSS_METRIC,
    loss_spatial_mode=LOSS_SPATIAL_MODE,
    loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
    loss_mask_buffer_px=0,  # coastal buffer disabled (see commented LOSS_MASK_BUFFER_PX in setup)
    # loss_mask_buffer_px=LOSS_MASK_BUFFER_PX,
    filter_fully_ocean_samples=FILTER_FULLY_OCEAN_SAMPLES,
    max_samples=MAX_SAMPLES,
    protect_existing_run=PROTECT_EXISTING_RUN,
    preview_loss_mask=PREVIEW_LOSS_MASK,
    preview_val_mask=PREVIEW_VAL_MASK,
    beta=BETA,
    lambda_perc=LAMBDA_PERC,
    perceptual_backend=PERCEPTUAL_BACKEND,
    lpips_net=LPIPS_NET,
    perceptual_weights=PERCEPTUAL_WEIGHTS,
    perceptual_warmup_epochs=PERCEPTUAL_WARMUP_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    checkpoint_subdir=str(MODELS_DIR),
    isolate_runs=True,
    channel_names=CHANNEL_NAMES,
    viz_every=VIZ_EVERY,
    viz_samples=VIZ_SAMPLES,
    viz_seed=VIZ_SEED,
    viz_residual_metric=VIZ_RESIDUAL_METRIC,
    viz_gaussian_circles=VIZ_GAUSSIAN_CIRCLES,
    viz_gaussian_circle_radii=VIZ_GAUSSIAN_CIRCLE_RADII,
    viz_gaussian_inner_circle_radii=VIZ_GAUSSIAN_INNER_CIRCLE_RADII,
    viz_gaussian_fig_scale=VIZ_GAUSSIAN_FIG_SCALE,
    show_inline_plots=SHOW_INLINE_PLOTS,
    val_split=VAL_SPLIT,
    val_split_mode=VAL_SPLIT_MODE,
    val_split_seed=VAL_SPLIT_SEED,
    val_reconstruction_metric=VAL_RECONSTRUCTION_METRIC,
    val_data_prefix=None if MERGE_VAL_EXPORT else VAL_DATA_PREFIX,
    merge_val_export_prefix=VAL_DATA_PREFIX if MERGE_VAL_EXPORT else None,
)

(_data, _mask, _day, _hour, _, _), _val_tensors = load_notebook_training_arrays(train_config)
_n_total = len(_data)
if train_config.val_data_prefix:
    _n_val = len(_val_tensors[0])
else:
    _n_val = None
_train_idx, _val_idx = resolve_train_val_indices(
    train_config, _n_total, n_val=_n_val
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
config_path.write_text(json.dumps(asdict(train_config), indent=2, default=str))
print(f"Run directory: {RUN_DIR}")
print(f"Wrote config:  {config_path}")
print(f"  data: {DATA_PREFIX} | iD={iD} | loss_metric={LOSS_METRIC}")
if train_config.merge_val_export_prefix and not train_config.val_data_prefix:
    split_note = (
        f"merged {VAL_SPLIT_MODE} split ({VAL_SPLIT:.0%} val, seed={VAL_SPLIT_SEED}): "
        f"{DATA_PREFIX} + {VAL_DATA_PREFIX}"
    )
elif train_config.val_data_prefix:
    split_note = f"export-time split (val from {VAL_DATA_PREFIX})"
else:
    split_note = f"{VAL_SPLIT_MODE} split on {DATA_PREFIX}"
print(
    f"  sequences: {_n_total} total | "
    f"train {len(_train_idx)} | val {len(_val_idx)} ({split_note})"
)
if LOSS_MODE == "l1":
    print("  loss mode: L1-only (no perceptual / discriminator)")
elif LOSS_MODE == "adv_no":
    perc_name = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"
    if PERCEPTUAL_WARMUP_EPOCHS == 0 and WARMUP_EPOCHS == 0:
        print(f"  loss mode: adv-NO — L1 + {perc_name} + adversarial from epoch 1")
    else:
        print(
            f"  loss mode: adv-NO — L1 through ep {PERCEPTUAL_WARMUP_EPOCHS}, "
            f"+{perc_name} from ep {PERCEPTUAL_WARMUP_EPOCHS + 1}, "
            f"+adv from ep {WARMUP_EPOCHS + 1}"
        )
else:
    print(f"  loss mode: {LOSS_MODE!r}")
if LOSS_SPATIAL_MODE == "full":
    spatial_note = "uniform over whole valid field"
elif LOSS_SPATIAL_MODE == "gaussian_center":
    spatial_note = f"gaussian_center (σ={LOSS_GAUSSIAN_SIGMA}) — state⊙G"
else:
    spatial_note = LOSS_SPATIAL_MODE
print(f"  loss spatial: {spatial_note}")
print(f"  rollout: best_model.pth + norm_stats.pt → notebook 05 gaussian")
if VIZ_GAUSSIAN_CIRCLES:
    print(f"  gauss-circle viz: r={VIZ_GAUSSIAN_CIRCLE_RADII} -> viz/samples_gauss_circles/")

In [ ]:
import numpy as np
from src.train.adv_no_train import _run_has_artifacts
from src.train.fix_window_train import SimulationDataset
from src.train.fix_window_viz import (
    fixed_viz_indices,
    save_loss_mask_preview,
    save_validation_mask_preview,
)

assert "train_config" in globals(), "Run the train config cell above first."
_mask_preview = np.load(f"{DATA_PREFIX}_mask.npy", mmap_mode="r")
land_frac = float(_mask_preview.mean())

if PROTECT_EXISTING_RUN and _run_has_artifacts(RUN_DIR, RUN_NAME):
    raise FileExistsError(
        f"Run already exists: {RUN_DIR}\n"
        "Change RUN_TAG to a new name before preview / training."
    )
elif _run_has_artifacts(RUN_DIR, RUN_NAME):
    print(f"Note: {RUN_DIR} exists — preview/training will overwrite prior results.")

if PREVIEW_LOSS_MASK:
    preview_path = save_loss_mask_preview(
        DATA_PREFIX,
        RUN_DIR / "viz" / "loss_mask_preview.png",
        sample_index=0,
        loss_spatial_mode=LOSS_SPATIAL_MODE,
        loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
        loss_mask_buffer_px=0,  # coastal buffer disabled
        # loss_mask_buffer_px=LOSS_MASK_BUFFER_PX,
        channel_names=CHANNEL_NAMES,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Loss mask preview: {preview_path}")
    if LOSS_SPATIAL_MODE == "full":
        if land_frac > 0:
            print(
                f"  full mode on ocean: uniform L1/perceptual/adv over valid pixels "
                f"(land {land_frac:.1%} masked out)"
            )
        else:
            print("  full mode: G=1 everywhere — loss treats all valid pixels equally")
    else:
        print(f"  gaussian_center: σ={LOSS_GAUSSIAN_SIGMA}px — center weighted more than edges")
else:
    print("Loss mask preview skipped (PREVIEW_LOSS_MASK=False)")

if PREVIEW_VAL_MASK:
    (_data, _mask, _day, _hour, _, _), _val_tensors = load_notebook_training_arrays(train_config)
    if train_config.val_data_prefix:
        _val_data, _val_mask, _val_day, _val_hour = _val_tensors[:4]
    else:
        _val_data = _val_mask = _val_day = _val_hour = None
    (
        _tr_data,
        _tr_mask,
        _tr_day,
        _tr_hour,
        _va_data,
        _va_mask,
        _va_day,
        _va_hour,
        _train_idx,
        _val_idx,
    ) = prepare_train_val_tensors(
        _data,
        _mask,
        _day,
        _hour,
        train_config,
        val_data=_val_data,
        val_mask=_val_mask,
        val_day=_val_day,
        val_hour=_val_hour,
    )
    _train_set = SimulationDataset(_tr_data, _tr_mask, _tr_day, _tr_hour, iD=iD)
    _val_set = SimulationDataset(
        _va_data,
        _va_mask,
        _va_day,
        _va_hour,
        mean=_train_set.mean,
        std=_train_set.std,
        iD=iD,
    )
    _viz_idx = fixed_viz_indices(PREVIEW_VAL_MASK_SAMPLES, len(_val_set), VIZ_SEED)
    _global_viz_idx = [int(_val_idx[i]) for i in _viz_idx]
    val_mask_path = save_validation_mask_preview(
        _val_set,
        RUN_DIR / "viz" / "val_mask_preview.png",
        sample_indices=_viz_idx,
        global_sample_indices=_global_viz_idx,
        channel_names=CHANNEL_NAMES,
        loss_mask_buffer_px=0,  # coastal buffer disabled
        # loss_mask_buffer_px=LOSS_MASK_BUFFER_PX,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Validation mask preview: {val_mask_path}")
    print(f"  val indices (in val set): {_viz_idx}")
    print(f"  global sequence indices: {_global_viz_idx}")
else:
    print("Validation mask preview skipped (PREVIEW_VAL_MASK=False)")

## Train

Foreground by default (`DETACHED = False`). Set `DETACHED = True` to launch `scripts/train_adv_no_fix_window.py` in the background (config must exist under `Magic_window/Models/adv_no/{RUN_NAME}/`).

In [ ]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_path = LOG_DIR / f"train_{RUN_NAME}.log"

if DETACHED:
    cmd = [
        sys.executable,
        str(ROOT / "scripts/train_adv_no_fix_window.py"),
        "--run-name",
        RUN_NAME,
        "--config",
        str(config_path),
    ]
    with open(log_path, "w") as log_file:
        proc = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            cwd=ROOT,
        )
    print(f"Started background adv-NO training PID {proc.pid}")
    print(f"Log: {log_path}")
    print(f"Monitor: tail -f {log_path}")
    print("Stop with: kill", proc.pid)
else:
    class _Tee:
        def __init__(self, *streams):
            self.streams = streams

        def write(self, data):
            for stream in self.streams:
                stream.write(data)
                stream.flush()

        def flush(self):
            for stream in self.streams:
                stream.flush()

    print(f"Foreground training — log: {log_path}")
    print("Stop anytime: Kernel → Interrupt (or ■ button)\n")

    _stdout = sys.stdout
    with open(log_path, "w") as log_file:
        sys.stdout = _Tee(_stdout, log_file)
        try:
            run_adv_no_training(train_config)
        except KeyboardInterrupt:
            print("\nNotebook interrupt received — training stopped.")
        finally:
            sys.stdout = _stdout

    print(f"\nLog saved: {log_path}")

## Loss curves (after epoch 1+)

In [ ]:
import matplotlib.pyplot as plt
import torch

history_path = RUN_DIR / "history" / f"training_history_{RUN_NAME}.pt"
state_path = RUN_DIR / f"train_state_{RUN_NAME}.json"
plot_path = RUN_DIR / "viz" / f"training_loss_curve_{RUN_NAME}.png"

if not history_path.exists():
    msg = [f"No history yet: {history_path}"]
    if state_path.exists():
        state = json.loads(state_path.read_text())
        msg.append(
            f"Training status: {state.get('status')} | "
            f"last completed epoch: {state.get('last_completed_epoch', 0)}/{state.get('epochs_planned', '?')}"
        )
    if log_path.exists():
        tail = log_path.read_text().splitlines()[-5:]
        msg.append("Recent log lines:")
        msg.extend(f"  {line}" for line in tail)
    msg.append("Re-run this cell after epoch 1 finishes.")
    print("\n".join(msg))
else:
    hist = torch.load(history_path, map_location="cpu", weights_only=False)
    epochs = hist["epoch"]

    if not epochs:
        print("History file exists but is empty. Wait for epoch 1 to finish, then re-run.")
    else:
        recon_label = "speed L1" if LOSS_METRIC == "speed" else "uv L1"
        fig, ax = plt.subplots(1, 3, figsize=(14, 4))
        ax[0].plot(epochs, hist["train_l1"], label=f"Train {recon_label}")
        ax[0].plot(epochs, hist["val_l1"], label=f"Val {recon_label}")
        ax[0].set_title(f"Reconstruction ({recon_label})")
        ax[0].legend()
        ax[0].grid(True, alpha=0.3)

        use_adv_no = LOSS_MODE == "adv_no"
        perc_warmup = PERCEPTUAL_WARMUP_EPOCHS if use_adv_no else 0
        adv_warmup = WARMUP_EPOCHS if use_adv_no else 0
        perc_title = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"

        if use_adv_no and any(hist.get("train_perceptual", [])):
            ax[1].plot(epochs, hist["train_perceptual"], label="Train perc")
            ax[1].plot(epochs, hist["val_perceptual"], label="Val perc")
            if perc_warmup > 0:
                ax[1].axvline(perc_warmup + 0.5, color="#2ca02c", linestyle=":", alpha=0.7)
            ax[1].set_title(f"Perceptual ({perc_title})")
            ax[1].legend()
            ax[1].grid(True, alpha=0.3)
        else:
            ax[1].set_title("Perceptual (starts after warmup)")
            ax[1].axis("off")

        if use_adv_no and any(hist.get("train_adv", [])):
            ax[2].plot(epochs, hist["train_adv"], label="Train adv")
            ax[2].plot(epochs, hist["train_d"], label="Train D")
            if adv_warmup > perc_warmup:
                ax[2].axvline(adv_warmup + 0.5, color="#888888", linestyle=":", alpha=0.7)
            ax[2].set_title("Adversarial terms")
            ax[2].legend()
            ax[2].grid(True, alpha=0.3)
        else:
            ax[2].set_title("Adversarial (starts after warmup)")
            ax[2].axis("off")

        plt.tight_layout()
        plot_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        if SHOW_INLINE_PLOTS:
            plt.show()
        else:
            plt.close(fig)
        print(f"Best val {recon_label}: {hist['best_val_loss']:.6f} at epoch {hist['best_epoch']}")
        print(f"Saved plot: {plot_path}")
        update_comparison_index(
            MODELS_DIR,
            run_name_value=RUN_NAME,
            export_key=EXPORT_KEY,
            run_dir_path=RUN_DIR,
            dataset_stats=dataset_stats if "dataset_stats" in globals() else None,
            training_summary={
                "best_val_loss": float(hist["best_val_loss"]),
                "best_epoch": int(hist["best_epoch"]),
                "epochs_completed": int(epochs[-1]) if epochs else 0,
            },
        )

## Compare weight-decay ablation

Run this after training arms **A/B/C** (`WEIGHT_DECAY` = 0, 1e-4, 3e-4). Overlays val curves and saves `PNG/ablation_weight_decay_{EXPORT_KEY}.png`.


In [ ]:
# Compare weight-decay ablation runs (run after arms A/B/C finish)
import matplotlib.pyplot as plt
import torch

from run_registry import GAUSS_RUN_TAG, run_dir, run_name, weight_decay_ablation_tags

assert "EXPORT_KEY" in globals(), "Run the setup cell first."

_base = BASE_RUN_TAG if "BASE_RUN_TAG" in globals() else GAUSS_RUN_TAG
_ablation_arms = weight_decay_ablation_tags(_base)
_rows = []
fig, ax = plt.subplots(figsize=(9, 4.5))
ylabel = "val RMSE"

for wd, tag in _ablation_arms:
    rd = run_dir(MODELS_DIR, EXPORT_KEY, dt=DT, run_tag=tag)
    run_nm = run_name(EXPORT_KEY, dt=DT, run_tag=tag)
    hist_path = rd / "history" / f"training_history_{run_nm}.pt"
    if not hist_path.exists():
        print(f"skip wd={wd:g}: missing {hist_path}")
        continue
    hist = torch.load(hist_path, map_location="cpu", weights_only=False)
    if hist.get("val_rmse"):
        val_y = hist["val_rmse"]
        ylabel = "val RMSE"
    else:
        val_y = hist.get("val_l1") or hist.get("val") or []
    epochs = hist.get("epoch") or list(range(1, len(val_y) + 1))
    best_i = int(min(range(len(val_y)), key=lambda i: val_y[i]))
    best_ep = int(hist.get("best_epoch") or epochs[best_i])
    _rows.append((wd, tag, float(val_y[best_i]), best_ep))
    ax.plot(epochs[: len(val_y)], val_y, linewidth=1.8, label=f"wd={wd:g}  best={val_y[best_i]:.4f} @ ep {best_ep}")

if not _rows:
    raise FileNotFoundError(
        "No ablation histories found. Train with WEIGHT_DECAY in {0, 1e-4, 3e-4} and "
        "ABLATION_SUFFIX_RUN_TAG=True first."
    )

ax.set_xlabel("epoch")
ax.set_ylabel(ylabel)
ax.set_title(f"Weight-decay ablation | {EXPORT_KEY} | val split seed={VAL_SPLIT_SEED}")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out_png = PNG_DIR / f"ablation_weight_decay_{EXPORT_KEY}.png"
fig.savefig(out_png, dpi=140, bbox_inches="tight")
plt.show()
print("Saved", out_png)
print("\nBest val metric by arm:")
for wd, tag, best_val, best_ep in sorted(_rows):
    print(f"  wd={wd:g:>8}  {best_val:.6f}  epoch {best_ep}  ({tag})")


## Next: rollout evaluation

After training completes, open [05_evaluate_adv_no_magic_window_rollout.ipynb](05_evaluate_adv_no_magic_window_rollout.ipynb).

It loads `{RUN_DIR}/best_model.pth` and saves rollout metrics to `Magic_window/Test&Rollout/{RUN_NAME}/rollout_results.json`. Set the same `EXPORT_KEY` in [05_evaluate_adv_no_magic_window_rollout_gaussian.ipynb](05_evaluate_adv_no_magic_window_rollout_gaussian.ipynb).

In [ ]:
print(f"Model dir: {RUN_DIR}")
print(f"  best_model: {RUN_DIR / 'best_model.pth'}")
print(f"  norm_stats: {RUN_DIR / 'norm_stats.pt'}")
print("Open 05_evaluate_adv_no_magic_window_rollout.ipynb when training is done.")